# ICL-PILOT Direct Generation Colab

This notebook runs the repo's native generation pipeline directly with `transformers`.

It keeps the existing experimental design fixed:
- the same frozen SLI-TD roster
- the same counterbalanced story schedule
- the same canonical ENNI story spines
- the same two-stage workflow: stage 1 TD-like story, then stage 2 SLI-like transformation

The only difference from the NeMo Data Designer notebook is the execution layer: here we build prompts and call the model ourselves.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/shamira-venturini/ICL-PILOT.git"
REPO_DIR = Path("/content/ICL-PILOT")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

%cd /content/ICL-PILOT

In [ ]:
%pip -q install -U pip setuptools wheel
%pip -q install -e . transformers accelerate bitsandbytes huggingface_hub sentencepiece

## Model And Run Settings

The report names `Meta-Llama-3.2-8B-Instruct` as the generation model. On Hugging Face, the usual repo id is `meta-llama/Llama-3.2-8B-Instruct`. If the model is gated in your environment, set `HF_TOKEN` first.

In [ ]:
import os
from getpass import getpass

# Uncomment if needed for gated model access.
# os.environ["HF_TOKEN"] = getpass("HF_TOKEN: ")

MODEL_NAME = "meta-llama/Llama-3.2-8B-Instruct"
SEED = 42
TEMPERATURE = 0.7
DO_SAMPLE = True
MAX_NEW_TOKENS_STAGE1 = 180
MAX_NEW_TOKENS_STAGE2 = 180
N_REPLICATES = 2
ROW_LIMIT = 6
SAVE_EVERY = 5

print({
    "MODEL_NAME": MODEL_NAME,
    "SEED": SEED,
    "TEMPERATURE": TEMPERATURE,
    "N_REPLICATES": N_REPLICATES,
    "ROW_LIMIT": ROW_LIMIT,
})

In [ ]:
import random
import numpy as np
import torch

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print({
    "cuda_available": torch.cuda.is_available(),
    "device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
})

In [ ]:
from icl_pilot.generation import build_generation_package

artifacts = build_generation_package(
    dev_measures_csv="phase2/measures/dev_measures.csv",
    severity_profile_csv="phase2/measures/severity_profile_banded_table.csv",
    counterbalance_csv="configs/generation/cross_story_counterbalancing.csv",
    output_dir="configs/generation/colab_generation_package_direct",
    n_replicates=N_REPLICATES,
)

artifacts

In [ ]:
import json
import pandas as pd

schedule_df = pd.read_csv(artifacts.schedule_csv)
story_reference = json.loads(Path("phase2/story_units/story_unit_reference.json").read_text())

def numbered_lines(lines):
    return "\n".join(f"{i + 1}. {line}" for i, line in enumerate(lines))

def bullet_lines(lines):
    if not lines:
        return "None"
    return "\n".join(f"- {line}" for line in lines)

run_df = schedule_df.copy()
run_df["story_title"] = run_df["target_story"].map(lambda story_id: story_reference[story_id]["title"])
run_df["story_units_text"] = run_df["target_story"].map(
    lambda story_id: numbered_lines(story_reference[story_id]["units"])
)
run_df["content_invariants_text"] = run_df["target_story"].map(
    lambda story_id: bullet_lines(story_reference[story_id].get("content_invariants", []))
)
run_df["optional_content_text"] = run_df["target_story"].map(
    lambda story_id: bullet_lines(story_reference[story_id].get("optional_content", []))
)
run_df["prompt_constraints_text"] = run_df["target_story"].map(
    lambda story_id: bullet_lines(story_reference[story_id].get("prompt_constraints", []))
)

if ROW_LIMIT is not None:
    run_df = run_df.head(ROW_LIMIT).copy()

print(f"Rows to generate: {len(run_df)}")
run_df[[
    "bundle_id",
    "replicate_id",
    "target_story",
    "story_title",
    "sli_child_id",
    "td_child_id",
    "prompt_td_story_e1",
    "prompt_sli_story_e2",
]].head()

## Load The Model

The default path tries 4-bit loading on GPU so the 8B model is more likely to fit on Colab. If you have a larger GPU, you can simplify this cell.

In [ ]:
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

if os.environ.get("HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

use_4bit = torch.cuda.is_available()
quant_config = None
model_kwargs = {
    "device_map": "auto",
}

if use_4bit:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16,
    )
    model_kwargs["quantization_config"] = quant_config
else:
    model_kwargs["torch_dtype"] = torch.float32

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)
model.eval()

print({
    "loaded_model": MODEL_NAME,
    "use_4bit": use_4bit,
    "pad_token": tokenizer.pad_token,
})

In [ ]:
def stage1_messages(row):
    return [
        {
            "role": "system",
            "content": (
                "You write short child-like ENNI narratives. Preserve the picture-supported event order, "
                "allow small surface variation, and do not add new major plot events."
            ),
        },
        {
            "role": "user",
            "content": f"""Generate a TD-like child retell for the ENNI story below.

Target story: {row.target_story} ({row.story_title})
Target story slot in the schedule: {row.target_story}
Cross-story TD exemplar slot: {row.prompt_td_story_e1}
Replicate id: {row.replicate_id}

Canonical story spine:
{row.story_units_text}

Content that should stay stable:
{row.content_invariants_text}

Optional details:
{row.optional_content_text}

Constraints:
{row.prompt_constraints_text}

Write one short child-like narrative.
Keep the story events in order.
Do not add new major events.
Do not write an adult-style polished retell.
Return only the narrative text.
""",
        },
    ]


def stage2_messages(row, stage1_text):
    return [
        {
            "role": "system",
            "content": (
                "You transform child narratives while preserving the underlying ENNI events and order. "
                "Make the output clinically plausible rather than caricatured."
            ),
        },
        {
            "role": "user",
            "content": f"""Transform the TD-like story below so it better matches the target SLI child profile.

Target story: {row.target_story} ({row.story_title})
Target child age: {row.sli_age}
Target severity band: {row.sli_severity_band}
Target profile label: {row.sli_profile_label}
Cross-story SLI exemplar slot: {row.prompt_sli_story_e2}

Canonical story spine:
{row.story_units_text}

Content that should stay stable:
{row.content_invariants_text}

Original TD-like story:
{stage1_text}

Instructions:
- Preserve the same major events and event order.
- Keep the narration child-like.
- Allow shorter clauses, simpler structure, clinically plausible omissions of optional details, and profile-consistent weakness.
- Avoid bizarre errors, random topic drift, or new plot events.
- Return only the transformed narrative text.
""",
        },
    ]


In [ ]:
def generate_from_messages(messages, max_new_tokens=180):
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {key: value.to(model.device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=DO_SAMPLE,
            temperature=TEMPERATURE,
            top_p=0.95,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return text.strip()


In [ ]:
example_row = run_df.iloc[0]
stage1_preview_prompt = tokenizer.apply_chat_template(
    stage1_messages(example_row),
    tokenize=False,
    add_generation_prompt=True,
)
print(stage1_preview_prompt[:4000])

In [ ]:
from tqdm.auto import tqdm

out_path = Path("data/generated/direct_generation/four_year_old_story_generations.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)

rows = []
for row in tqdm(list(run_df.itertuples(index=False)), total=len(run_df)):
    stage1_text = generate_from_messages(
        stage1_messages(row),
        max_new_tokens=MAX_NEW_TOKENS_STAGE1,
    )
    stage2_text = generate_from_messages(
        stage2_messages(row, stage1_text),
        max_new_tokens=MAX_NEW_TOKENS_STAGE2,
    )

    rows.append({
        "bundle_id": row.bundle_id,
        "pair_id": row.pair_id,
        "replicate_id": row.replicate_id,
        "story_slot_order": row.story_slot_order,
        "target_story": row.target_story,
        "story_title": row.story_title,
        "sli_child_id": row.sli_child_id,
        "td_child_id": row.td_child_id,
        "sli_age": row.sli_age,
        "sli_severity_band": row.sli_severity_band,
        "sli_profile_label": row.sli_profile_label,
        "prompt_td_story_e1": row.prompt_td_story_e1,
        "prompt_sli_story_e2": row.prompt_sli_story_e2,
        "story_units_text": row.story_units_text,
        "stage1_td_story": stage1_text,
        "stage2_sli_story": stage2_text,
    })

    if len(rows) % SAVE_EVERY == 0:
        pd.DataFrame(rows).to_csv(out_path, index=False)

generated_df = pd.DataFrame(rows)
generated_df.to_csv(out_path, index=False)
print(f"Saved {len(generated_df)} rows to {out_path}")
generated_df[["bundle_id", "target_story", "stage1_td_story", "stage2_sli_story"]].head()

## Next Step

Once the story-level rows look good, you can aggregate them by `bundle_id`, convert them into transcript-like bundle files, and then feed them into the same downstream annotation and evaluation workflow used elsewhere in the repo.